What is the longest match recorded in terms of duration?

In [1]:
import pandas as pd
from pathlib import Path

In [3]:
# Define the project and data paths

PROJECT_ROOT = Path.cwd().parents[1]

DATA_ROOT = PROJECT_ROOT / "tennis_data"

data_path = DATA_ROOT / "Data"

In [22]:
# Define the exact path to the processed Time dataset

time_path = (
    PROJECT_ROOT
    / "tennis_data_local_backup"
    / "processed"
    / "time_clean.parquet"
)

In [9]:
time = pd.read_parquet(time_path)

time.head()

,match_id,period_1,period_2,period_3,period_4,period_5,current_period_start_timestamp,snapshot_date
0,11974053,NaN,NaN,NaN,NaN,NaN,NaN,2024-02-01
1,11974066,NaN,NaN,NaN,NaN,NaN,NaN,2024-02-01
2,11998445,3259.0,2639.0,4202.0,NaN,NaN,1.706817e+09,2024-02-01
3,11998446,2488.0,2375.0,NaN,NaN,NaN,1.706804e+09,2024-02-01
4,11998447,3741.0,1913.0,NaN,NaN,NaN,1.706798e+09,2024-02-01


In [10]:
# Inspect the Time dataset

print("Shape:", time.shape)

print("\nColumns:")
print(time.columns.tolist())

print("\nData types:")
print(time.dtypes)

time.head()

Shape: (35671, 8)

Columns:
['match_id', 'period_1', 'period_2', 'period_3', 'period_4', 'period_5', 'current_period_start_timestamp', 'snapshot_date']

Data types:
match_id                            int64
period_1                          float64
period_2                          float64
period_3                          float64
period_4                          float64
period_5                          float64
current_period_start_timestamp    float64
snapshot_date                      object
dtype: object


,match_id,period_1,period_2,period_3,period_4,period_5,current_period_start_timestamp,snapshot_date
0,11974053,NaN,NaN,NaN,NaN,NaN,NaN,2024-02-01
1,11974066,NaN,NaN,NaN,NaN,NaN,NaN,2024-02-01
2,11998445,3259.0,2639.0,4202.0,NaN,NaN,1.706817e+09,2024-02-01
3,11998446,2488.0,2375.0,NaN,NaN,NaN,1.706804e+09,2024-02-01
4,11998447,3741.0,1913.0,NaN,NaN,NaN,1.706798e+09,2024-02-01


In [ ]:
# Check whether matches appear in multiple snapshots

# Convert snapshot_date to a real datetime column
time["snapshot_date"] = pd.to_datetime(time["snapshot_date"])

# Count how many rows exist for each match
match_counts = time["match_id"].value_counts()

print("Total rows:", len(time))
print("Unique matches:", time["match_id"].nunique())
print("Matches appearing more than once:", (match_counts > 1).sum())

match_counts.head(10)

Total rows: 35671
Unique matches: 16873
Matches appearing more than once: 16343


match_id
12063582    4
12063583    4
12063587    4
12063588    4
12063589    4
12063599    4
12063611    4
12063615    4
12084420    4
12086016    4
Name: count, dtype: int64

In [12]:
# Inspect one match that appears in multiple snapshots

example_match_id = match_counts.index[0]

time[
    time["match_id"] == example_match_id
].sort_values("snapshot_date")

,match_id,period_1,period_2,period_3,period_4,period_5,current_period_start_timestamp,snapshot_date
5793,12063582,157690.0,4654.0,NaN,NaN,NaN,1.707981e+09,2024-02-12
6171,12063582,157690.0,4654.0,NaN,NaN,NaN,1.707981e+09,2024-02-13
6602,12063582,157690.0,4654.0,NaN,NaN,NaN,1.707981e+09,2024-02-14
7084,12063582,157690.0,4654.0,NaN,NaN,NaN,1.707981e+09,2024-02-15


In [13]:
# Keep the latest snapshot for each match

time_latest = (
    time
    .sort_values("snapshot_date")
    .drop_duplicates(subset="match_id", keep="last")
    .copy()
)

print("Rows before:", len(time))
print("Rows after:", len(time_latest))

Rows before: 35671
Rows after: 16873


In [14]:
# Inspect the distribution and largest values for each set duration

period_columns = [
    "period_1",
    "period_2",
    "period_3",
    "period_4",
    "period_5"
]

print(time_latest[period_columns].describe())

print("\nLargest values in each period:")
for col in period_columns:
    print(f"\n{col}")
    print(
        time_latest[
            ["match_id", col]
        ]
        .dropna()
        .sort_values(col, ascending=False)
        .head(10)
        .to_string(index=False)
    )

            period_1       period_2       period_3  period_4  period_5
count   11089.000000   11071.000000    3391.000000       0.0       0.0
mean     2866.858418    3084.054738    3175.481569       NaN       NaN
std      4888.519816    5141.821530    6262.163425       NaN       NaN
min         2.000000  -16200.000000       0.000000       NaN       NaN
25%      2043.000000    2116.000000    1957.500000       NaN       NaN
50%      2515.000000    2606.000000    2602.000000       NaN       NaN
75%      3149.000000    3255.000000    3292.000000       NaN       NaN
max    172606.000000  169438.000000  152072.000000       NaN       NaN

Largest values in each period:

period_1
 match_id  period_1
 12121829  172606.0
 12063611  167352.0
 12063589  159242.0
 12063587  159144.0
 12063588  158918.0
 12063615  158909.0
 12063583  158245.0
 12063582  157690.0
 12189039   88279.0
 12054312   86389.0

period_2
 match_id  period_2
 12063611  169438.0
 12063587  161086.0
 12195781  125538.0
 12069269

In [15]:
# Remove rows where period_1 is missing
# period_1 is required because every valid match must have at least one played set

time_q4 = time_latest[~time_latest["period_1"].isna()].copy()

# Remove outliers from period_1 and period_2 using the IQR method

for col in ["period_1", "period_2"]:
    Q1 = time_q4[col].quantile(0.25)
    Q3 = time_q4[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR

    time_q4 = time_q4[
        (time_q4[col] > lower_fence) &
        (time_q4[col] < upper_fence)
    ]

print("Rows remaining:", len(time_q4))

Rows remaining: 10531


In [16]:
# Remove outliers from period_3 using the IQR method
# Keep NaN values because a third set is not always played

Q1 = time_q4["period_3"].quantile(0.25)
Q3 = time_q4["period_3"].quantile(0.75)

IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

time_q4 = time_q4[
    (
        (time_q4["period_3"] > lower_fence) &
        (time_q4["period_3"] < upper_fence)
    )
    |
    (time_q4["period_3"].isna())
]

print("Rows remaining:", len(time_q4))

Rows remaining: 10453


In [17]:
# Calculate total match duration
# NaN in period_3 is treated as 0 because the match ended in two sets

time_q4["total_time"] = (
    time_q4[["period_1", "period_2", "period_3"]]
    .fillna(0)
    .sum(axis=1)
)

time_q4[
    ["match_id", "period_1", "period_2", "period_3", "total_time"]
].head()

,match_id,period_1,period_2,period_3,total_time
182,12021562,1853.0,2723.0,NaN,4576.0
183,12021566,3549.0,4159.0,3407.0,11115.0
184,12021581,2844.0,3652.0,3550.0,10046.0
185,12021589,2618.0,2623.0,NaN,5241.0
186,12021590,3064.0,1382.0,NaN,4446.0


In [18]:
# Convert total match duration from seconds to hours:minutes

time_q4["total_time_hm"] = time_q4["total_time"].apply(
    lambda x: f"{int(x // 3600)}:{int((x % 3600) // 60):02d}"
)

time_q4[
    ["match_id", "total_time", "total_time_hm"]
].head()

,match_id,total_time,total_time_hm
182,12021562,4576.0,1:16
183,12021566,11115.0,3:05
184,12021581,10046.0,2:47
185,12021589,5241.0,1:27
186,12021590,4446.0,1:14


In [19]:
# Find the 5 longest matches after cleaning

final_time_info = (
    time_q4[
        ["match_id", "total_time", "total_time_hm"]
    ]
    .sort_values(
        by="total_time",
        ascending=False
    )
    .head(5)
)

final_time_info

,match_id,total_time,total_time_hm
662,12024245,13204.0,3:40
3623,12046963,12914.0,3:35
7768,12072523,12762.0,3:32
29465,12171560,12600.0,3:30
14901,12088096,12575.0,3:29


In [20]:
# Load the player tables

home_team = pd.read_parquet(
    PROJECT_ROOT
    / "tennis_data_local_backup"
    / "processed"
    / "home_team.parquet"
)

away_team = pd.read_parquet(
    PROJECT_ROOT
    / "tennis_data_local_backup"
    / "processed"
    / "away_team.parquet"
)

In [21]:
# Final answer for Question 4

longest_match = final_time_info.iloc[0]

print("Longest match ID:", longest_match["match_id"])
print("Longest match duration:", longest_match["total_time_hm"])

Longest match ID: 12024245
Longest match duration: 3:40
